[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/08_Your_First_Knowledge_Graph.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 🚀 你的第一张知识图谱

## 概览

本 notebook 带你从一个简单文档出发，创建你的第一张知识图谱。你将走完从摄取文件到可视化图谱的完整端到端工作流——并且每一步都消费上一步的真实输出。

> [!TIP]
> 如果你是 Semantica 新手，这里是最好的起点。不需要任何知识图谱基础！

**文档**：[API 参考](https://semantica.readthedocs.io/reference/kg/)

### 🎯 学习目标

- **理解工作流**：掌握 `文件 → 解析 → 抽取 → 建图 → 可视化` 流水线
- **摄取数据**：用 `FileIngestor` 加载文档
- **解析内容**：用 `DocumentParser` 提取文本
- **抽取知识**：用 `NERExtractor` 和 `RelationExtractor` 识别实体和关系
- **构建图谱**：用 `GraphBuilder` 构建图
- **可视化**：用 `KGVisualizer` 看你的图活起来

## 安装

从 PyPI 安装 Semantica：

```bash
pip install semantica
# 或安装全部可选依赖：
pip install semantica[all]
```

---

## 🔄 简单的端到端工作流

### 先看主线

前几篇各练了一件工具：02 摄取、03 解析、05 圈实体、06 画关系、07 组装成图。本篇是总装车间——把所有零件依次装上一条流水线，最后再加一件新零件（可视化），产出你的第一张图。

完整工作流共五步，每一步正好用上前面某一篇练过的本事：

1. **📥 摄取**——加载文件（02 篇的 `FileIngestor`）
2. **📄 解析**——提取文本（03 篇的 `DocumentParser`）
3. **⛏️ 抽取**——识别实体与关系（05、06 篇的两个抽取器）
4. **🕸️ 建图**——组装知识图谱（07 篇的 `GraphBuilder`）
5. **📊 可视化**——渲染并观察图（本篇新出场）

两点导读：其一，本篇每一步消费的都是上一步的真实输出，不是各自为政的独立演示——串起来才叫流水线。其二，第 4 步建出的图刻意不去重，`Apple Inc.` 会有三个节点；合并重复正是 07 篇的主题，两篇首尾相扣。

下面的代码 cell 逐步演示，且每个 cell 都可独立重跑：示例文件只在最后的可选清理 cell 里才会删除。

> [!TIP]
> **备选：使用 Semantica 框架类**
>
> 想要更简单的高层做法，可以用 `Semantica` 框架类一步编排所有步骤——相当于自动挡；本篇选择手动挡逐档教学：
>
> ```python
> from semantica.core import Semantica
>
> framework = Semantica()
> framework.initialize()
>
> result = framework.build_knowledge_base(
>     sources=["sample_document.txt"],
>     embeddings=True,
>     graph=True
> )
>
> framework.shutdown()
> ```
>
> 更多细节见[核心模块使用指南](../../../semantica/core/core_usage.md)。

---

## 📂 第 1 步：摄取文件

这一步用 `FileIngestor` 加载文档。摄取器支持 PDF、DOCX、TXT 等多种格式。写示例文件是幂等操作，这个 cell 随时可重跑。

In [ ]:
%pip install semantica

# spaCy models are distributed separately from the spaCy library. This lesson
# relies on the English model to recognize standalone places such as Cupertino.
import sys
import subprocess
import spacy

try:
    spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])


In [ ]:
from pathlib import Path

from semantica.ingest import FileIngestor

sample_text = """Apple Inc. is headquartered in Cupertino, California.
In 1976, Steve Jobs founded Apple Inc.
Tim Cook is the CEO of Apple Inc.
"""

sample_file = Path("sample_document.txt")
sample_file.write_text(sample_text)

print(f"File: {sample_file}")
print(f"Content length: {len(sample_text)} characters")

ingestor = FileIngestor()
file_object = ingestor.ingest_file(sample_file, read_content=True)
print(f"  File name: {file_object.name}")
print(f"  File type: {file_object.file_type}")
print(f"  Content available: {file_object.content is not None}")

## 📄 第 2 步：解析文档

文件摄取进来后，还要解析出文本内容。`DocumentParser.parse_document()` 把抽取到的文本放在 `"text"` 键下返回。

In [ ]:
from semantica.parse import DocumentParser

parser = DocumentParser()
parsed_document = parser.parse_document(str(sample_file))

parsed_content = parsed_document.get("text", "")
assert parsed_content.strip(), "Parsing produced no text — check the input file"

print(f"Parsed content length: {len(parsed_content)} characters")
print(f"Preview: {parsed_content[:120]}...")

## ⛏️ 第 3 步：抽取实体和关系

接下来从解析出的文本里抽取实体和关系。`NERExtractor` 识别人物、组织、地点和日期；`RelationExtractor` 在这些提及之间找关系。两者操作的都是**第 2 步解析出的内容**——不是原始字符串的副本。

In [ ]:
from semantica.semantic_extract import NERExtractor, RelationExtractor

ner_extractor = NERExtractor()
relation_extractor = RelationExtractor()

mentions = ner_extractor.extract(parsed_content)
relations = relation_extractor.extract(parsed_content, mentions)

print("Entity mentions:")
for mention in mentions:
    print(f"  {mention.text!r:<13} {mention.label:<7} span=[{mention.start_char}:{mention.end_char}]")

print("\nExtracted relations:")
for rel in relations:
    print(f"  {rel.subject.text!r} --{rel.predicate}--> {rel.object.text!r}")

## 🕸️ 第 4 步：构建知识图谱

用抽取出的实体和关系，通过 `GraphBuilder` 组装知识图谱。每个提及拿到一个图 ID，每条边都从 `Relation.subject` / `Relation.object` 的真实端点构建——和 07 篇第 2 步同一套做法。

> [!NOTE]
> 图里每个*提及*占一个节点，所以 `Apple Inc.` 会出现三次。把重复提及合并成一个规范实体，正是 [07_Building_Knowledge_Graphs.ipynb](./07_Building_Knowledge_Graphs.ipynb) 的主题。

In [ ]:
from semantica.kg import GraphBuilder

entities = []
span_to_id = {}
for i, mention in enumerate(mentions, 1):
    graph_id = f"e{i}"
    span_to_id[(mention.start_char, mention.end_char)] = graph_id
    entities.append({
        "id": graph_id,
        "type": mention.label,
        "name": mention.text,
        "properties": {},
    })

relationships = []
for rel in relations:
    source_id = span_to_id.get((rel.subject.start_char, rel.subject.end_char))
    target_id = span_to_id.get((rel.object.start_char, rel.object.end_char))
    if source_id is None or target_id is None:
        print(f"Skipping relation with unmapped endpoint: "
              f"{rel.subject.text!r} --{rel.predicate}--> {rel.object.text!r}")
        continue
    relationships.append({
        "source": source_id,
        "target": target_id,
        "type": rel.predicate,
        "properties": {},
    })

builder = GraphBuilder()
knowledge_graph = builder.build({"entities": entities, "relationships": relationships})

id_to_name = {entity["id"]: entity["name"] for entity in entities}

print(f"Nodes (entities): {len(knowledge_graph['entities'])}")
for entity in knowledge_graph["entities"]:
    print(f"  {entity['id']}: {entity['name']} ({entity['type']})")

print(f"\nEdges (relationships): {len(knowledge_graph['relationships'])}")
for relationship in knowledge_graph["relationships"]:
    print(f"  {id_to_name[relationship['source']]} "
          f"--{relationship['type']}--> {id_to_name[relationship['target']]}")

edges = {
    (id_to_name[r["source"]], r["type"], id_to_name[r["target"]])
    for r in knowledge_graph["relationships"]
}
assert ("Apple Inc.", "located_in", "Cupertino") in edges
assert ("Tim Cook", "works_for", "Apple Inc.") in edges

## 📊 第 5 步：可视化与分析

最后用 `KGVisualizer` 渲染知识图谱，观察它的结构。`visualize_network()` 直接接受 `GraphBuilder` 的结果，还能保存交互式 HTML 文件。

In [ ]:
from semantica.visualization import KGVisualizer

visualizer = KGVisualizer()
fig = visualizer.visualize_network(
    knowledge_graph, output="html", file_path="knowledge_graph.html"
)
print("Saved interactive visualization to knowledge_graph.html")

entity_types = {}
for entity in knowledge_graph["entities"]:
    entity_types[entity["type"]] = entity_types.get(entity["type"], 0) + 1

print("\nEntities by type:")
for entity_type, count in sorted(entity_types.items()):
    print(f"  - {entity_type}: {count}")

relationship_types = {}
for relationship in knowledge_graph["relationships"]:
    relationship_types[relationship["type"]] = (
        relationship_types.get(relationship["type"], 0) + 1
    )

print("\nRelationships by type:")
for relationship_type, count in sorted(relationship_types.items()):
    print(f"  - {relationship_type}: {count}")

fig

## 🧹 可选：清理

确认做完本 notebook 之后再运行这个 cell。前面的 cell 要读 `sample_document.txt`，在删除之前它们都能重跑。

In [ ]:
for path in [sample_file, Path("knowledge_graph.html")]:
    if path.exists():
        path.unlink()
        print(f"Removed {path}")

## 小结

### 回顾主线

一条流水线五个工位，每一步的输出就是下一步的输入：文件进、文本出；文本进、提及和关系出；提及和关系进、图出；图进、可视化出。刻意没做的一件事是去重——那是 07 篇的主题，也是你下一步该去的地方。

### 你做到了

你端到端建成了自己的第一张知识图谱：

- **FileIngestor** 加载了示例文档
- **DocumentParser** 在 `"text"` 键下返回了它的文本
- **NERExtractor** / **RelationExtractor** 从该文本产出了真实的提及和关系
- **GraphBuilder** 把它们组装成一张边来自真实关系端点的图
- **KGVisualizer** 把结果渲染成了交互式网络

下一步：去 [07_Building_Knowledge_Graphs.ipynb](./07_Building_Knowledge_Graphs.ipynb) 用 `EntityResolver` 合并重复提及，或在 Graph Analytics notebook 里探索图指标。